# ✍️ Reto práctico: Escribe tus propios prompts
### Zero-shot vs Few-shot Prompting

En este notebook **no hay respuestas de opción múltiple**. Vas a **escribir tus propios prompts**
para 6 tareas distintas, alternando entre Zero-shot y Few-shot.

El código no puede saber si tu prompt es "creativo" o "está bien redactado" — eso lo va a evaluar
tu profesor/a y tus compañeros en clase. Pero sí puede comprobar si tu prompt tiene la **estructura**
correcta de la técnica que te piden (por ejemplo, si un Few-shot realmente incluye ejemplos).

**Cómo se usa en clase:**
1. Cada estudiante resuelve los 6 ejercicios en su propia copia del notebook.
2. Ejecutan la celda de diagnóstico de cada ejercicio: les dice si la **estructura** es correcta.
3. En clase, el profesor proyecta 2 o 3 notebooks de estudiantes (o los revisa uno por uno) y
   discute en voz alta con el grupo si el **contenido** del prompt es claro, específico y cumpliría
   el objetivo.
4. Al final hay una **Ficha de corrección en clase** para completar entre todos.

---


## ⚙️ Configuración inicial (ejecuta esta celda primero, no la modifiques)

In [ ]:
import re
import unicodedata

resultados = {}

def normalizar(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto

def contar_ejemplos(texto):
    t = normalizar(texto)
    n_ejemplo = len(re.findall(r"ejemplo", t))
    n_flecha = len(re.findall(r"->|→", t))
    return n_ejemplo, n_flecha

def diagnostico_zero_shot(nombre, prompt):
    """Comprueba que un prompt Zero-shot sea una instrucción directa, sin ejemplos."""
    if prompt.strip() == "":
        print("✏️ Todavía no has escrito tu prompt.")
        resultados[nombre] = None
        return
    n_ejemplo, n_flecha = contar_ejemplos(prompt)
    tiene_instruccion = len(prompt.strip().split()) >= 6
    sin_ejemplos = n_ejemplo == 0 and n_flecha == 0

    print("── Diagnóstico estructural (Zero-shot) ──")
    print(f"{'✅' if tiene_instruccion else '⚠️'} Da una instrucción con suficiente detalle ({len(prompt.split())} palabras)")
    print(f"{'✅' if sin_ejemplos else '⚠️'} No incluye ejemplos previos (correcto en Zero-shot)")
    if not sin_ejemplos:
        print("   → Si incluiste ejemplos de entrada/salida, en realidad estás haciendo Few-shot, no Zero-shot.")
    print()
    print("📌 Esto NO evalúa si tu instrucción es clara o específica — eso se discute en clase.")
    resultados[nombre] = {"tipo": "zero-shot", "estructura_ok": tiene_instruccion and sin_ejemplos, "prompt": prompt}

def diagnostico_few_shot(nombre, prompt, min_ejemplos=2):
    """Comprueba que un prompt Few-shot incluya varios ejemplos entrada->salida y una tarea final."""
    if prompt.strip() == "":
        print("✏️ Todavía no has escrito tu prompt.")
        resultados[nombre] = None
        return
    n_ejemplo, n_flecha = contar_ejemplos(prompt)
    tiene_ejemplos = n_ejemplo >= min_ejemplos or n_flecha >= min_ejemplos
    t = normalizar(prompt)
    tiene_tarea_final = "tarea" in t or "ahora" in t or "nuevo" in t

    print("── Diagnóstico estructural (Few-shot) ──")
    print(f"{'✅' if tiene_ejemplos else '⚠️'} Incluye al menos {min_ejemplos} ejemplos de entrada→salida (detectados: {max(n_ejemplo, n_flecha)})")
    print(f"{'✅' if tiene_tarea_final else '⚠️'} Se distingue una tarea/entrada nueva al final, distinta de los ejemplos")
    if not tiene_ejemplos:
        print("   → Recuerda: Few-shot necesita ejemplos reales de entrada y salida, no solo la instrucción.")
    print()
    print("📌 Esto NO evalúa si tus ejemplos son claros o representativos — eso se discute en clase.")
    resultados[nombre] = {"tipo": "few-shot", "estructura_ok": tiene_ejemplos and tiene_tarea_final, "prompt": prompt}

print("Entorno listo. ¡Comienza a escribir tus prompts! 🚀")


---
## 1️⃣ Zero-shot · Clasificar tickets de soporte

**Tarea:** Quieres que un modelo clasifique un ticket de soporte técnico en **Alta**, **Media** o **Baja** urgencia, dando solo la instrucción, sin ejemplos previos.

Escribe un prompt Zero-shot que:
- Diga claramente qué tiene que hacer el modelo.
- Indique las tres categorías posibles.
- Deje claro dónde iría el texto del ticket (puedes usar algo como `[TICKET]` como marcador).

In [ ]:
# ESCRIBE TU PROMPT ZERO-SHOT AQUÍ
prompt_1 = """

"""

diagnostico_zero_shot("ejercicio_1", prompt_1)


**🗣️ Para discutir en clase:**
- ¿Tu instrucción sería igual de clara si la leyera alguien que no sabe nada del tema?
- ¿Qué pasaría si el modelo no supiera qué hacer si el ticket no encaja en ninguna categoría?

---
## 2️⃣ Few-shot · La misma tarea, pero exigiendo un formato exacto

**Tarea:** Ahora necesitas que la respuesta del modelo sea **siempre** un JSON con esta forma:
```
{"urgencia": "Alta", "motivo": "..."}
```

Escribe un prompt Few-shot con **al menos 2 ejemplos** de ticket → JSON de salida, y termina con un ticket nuevo sin resolver (una tarea).

In [ ]:
# ESCRIBE TU PROMPT FEW-SHOT AQUÍ
prompt_2 = """

"""

diagnostico_few_shot("ejercicio_2", prompt_2)


**🗣️ Para discutir en clase:**
- ¿Tus ejemplos muestran claramente el formato JSON exacto que quieres?
- ¿Elegiste ejemplos variados (una urgencia alta y una baja, por ejemplo) o los dos son parecidos?

---
## 3️⃣ Zero-shot · Resumir un texto en una sola frase

**Tarea:** Escribe un prompt Zero-shot que le pida al modelo resumir un párrafo en **una sola frase de máximo 20 palabras**. Usa `[TEXTO]` como marcador para el párrafo original.

In [ ]:
# ESCRIBE TU PROMPT ZERO-SHOT AQUÍ
prompt_3 = """

"""

diagnostico_zero_shot("ejercicio_3", prompt_3)


**🗣️ Para discutir en clase:**
- ¿Puede el modelo saber, solo con tu instrucción, qué es lo más importante del texto?
- ¿Tu límite de palabras es realista para cualquier párrafo, largo o corto?

---
## 4️⃣ Few-shot · Extraer datos de una reseña de producto

**Tarea:** Quieres convertir reseñas de clientes en datos estructurados: calificación (1 a 5) y sentimiento (Positivo/Negativo/Neutro).

Escribe un prompt Few-shot con **al menos 2 ejemplos** de reseña → salida estructurada, y termina con una reseña nueva sin resolver.

In [ ]:
# ESCRIBE TU PROMPT FEW-SHOT AQUÍ
prompt_4 = """

"""

diagnostico_few_shot("ejercicio_4", prompt_4)


**🗣️ Para discutir en clase:**
- ¿Tus ejemplos cubren casos distintos (una reseña buena y una mala)?
- ¿Es fácil ver, solo mirando tus ejemplos, exactamente qué campos debe devolver el modelo?

---
## 5️⃣ Zero-shot · Traducir una frase

**Tarea:** Escribe un prompt Zero-shot que pida traducir una frase de español a inglés, indicando que la traducción debe sonar natural (no literal palabra por palabra). Usa `[FRASE]` como marcador.

In [ ]:
# ESCRIBE TU PROMPT ZERO-SHOT AQUÍ
prompt_5 = """

"""

diagnostico_zero_shot("ejercicio_5", prompt_5)


**🗣️ Para discutir en clase:**
- ¿Cómo le explicarías a un modelo qué significa "sonar natural" sin dar un ejemplo?
- ¿En qué casos convendría más pasar esta tarea a Few-shot?

---
## 6️⃣ Few-shot · Cambiar un texto formal a un tono casual

**Tarea:** Quieres que el modelo reescriba frases formales en un tono casual y cercano, manteniendo el mismo significado.

Escribe un prompt Few-shot con **al menos 2 ejemplos** de frase formal → frase casual, y termina con una frase formal nueva sin resolver.

In [ ]:
# ESCRIBE TU PROMPT FEW-SHOT AQUÍ
prompt_6 = """

"""

diagnostico_few_shot("ejercicio_6", prompt_6)


**🗣️ Para discutir en clase:**
- ¿Tus dos ejemplos muestran el mismo "tipo" de cambio de tono, o son muy distintos entre sí?
- ¿El modelo podría copiar tus ejemplos literalmente en vez de generalizar el patrón? ¿Cómo lo evitarías?

---
## 📋 Resumen para la corrección en clase

Ejecuta esta celda al terminar los 6 ejercicios. Te da un resumen para proyectar o compartir con el
profesor, con el diagnóstico estructural de cada prompt. La calidad real de cada prompt se discute
y califica en clase, no aquí.

In [ ]:
print("=" * 50)
print("   RESUMEN PARA CORRECCIÓN EN CLASE")
print("=" * 50)

orden = ["ejercicio_1", "ejercicio_2", "ejercicio_3", "ejercicio_4", "ejercicio_5", "ejercicio_6"]
titulos = {
    "ejercicio_1": "1. Zero-shot · Clasificar tickets",
    "ejercicio_2": "2. Few-shot · Clasificar con formato JSON",
    "ejercicio_3": "3. Zero-shot · Resumir en una frase",
    "ejercicio_4": "4. Few-shot · Extraer datos de reseña",
    "ejercicio_5": "5. Zero-shot · Traducir una frase",
    "ejercicio_6": "6. Few-shot · Cambiar el tono",
}

completados = 0
estructura_ok = 0

for clave in orden:
    r = resultados.get(clave)
    print(f"\n{titulos[clave]}")
    if r is None:
        print("   Estado: ⬜ sin responder")
        continue
    completados += 1
    if r["estructura_ok"]:
        estructura_ok += 1
        print("   Estructura: ✅ correcta")
    else:
        print("   Estructura: ⚠️ revisar (ver diagnóstico de la celda correspondiente)")
    print(f"   Tipo esperado: {r['tipo']}")

print("\n" + "-" * 50)
print(f"Ejercicios completados: {completados} / 6")
print(f"Con estructura correcta: {estructura_ok} / 6")
print("-" * 50)
print("👉 Espacio para la nota del profesor / discusión en clase:")
print("   Claridad de la instrucción:      ___ / 5")
print("   Calidad y variedad de ejemplos:  ___ / 5")
print("   Formato de salida bien definido: ___ / 5")
print("   Comentarios: ______________________________________")
